# 13 — Color + MLP | Original & Masked
`MLPClassifier`. `RandomizedSearchCV` ile **sadece train üzerinde 3-fold** hiperparametre araması; `StandardScaler` CV pipeline içinde. Sonuç: validation Macro F1. 

In [1]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import BASE_DIR
from src.data_loader import load_dataset
FEATURE_DIR = BASE_DIR / "outputs" / "features"
RESULT_DIR = BASE_DIR / "outputs" / "models" / "deep_learning"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset(require_masks=True).reset_index(drop=True)
y = df['label'].to_numpy(dtype=str)
ids = df['image'].to_numpy(dtype=str)
classes = np.unique(y)
idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=.15, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(dev_idx, test_size=.15/.85, random_state=42, stratify=y[dev_idx])
assert len(set(train_idx)&set(val_idx)) == len(set(train_idx)&set(test_idx)) == len(set(val_idx)&set(test_idx)) == 0
print('Train:',len(train_idx),'Val:',len(val_idx),'Test (kullanılmayacak):',len(test_idx))
# ÖNEMLİ: Aynı lezyondan farklı görüntüler varsa lesion_id ile group split ayrıca uygulanmalı.


Train: 7009 Val: 1503 Test (kullanılmayacak): 1503


In [2]:
paths = {'original': FEATURE_DIR/'X_original_color_300x225.npy', 'masked':FEATURE_DIR/'X_masked_color_300x225.npy'}
ids_file = FEATURE_DIR/'image_ids_color_300x225.npy'
y_file = FEATURE_DIR/'y_labels_color_300x225.npy'
assert np.array_equal(np.load(ids_file,allow_pickle=False).astype(str), ids)
assert np.array_equal(np.load(y_file,allow_pickle=False).astype(str), y)
for p in paths.values():
    x0=np.load(p,mmap_mode='r',allow_pickle=False)
    assert x0.shape == (len(y),439), (p,x0.shape)
print('Color ID/etiket sırası doğrulandı.')


Color ID/etiket sırası doğrulandı.


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform
from joblib import dump
MODEL_NAME='MLP_Color'
N_ITER=12   # süreye göre 6 / 12 / 24
CV_FOLDS=3
SEARCH_JOBS=1 # 16 GB RAM için
base_model=Pipeline([('scale',StandardScaler()),('model',MLPClassifier(max_iter=240,early_stopping=True,validation_fraction=.15,n_iter_no_change=15,random_state=42))])
param_space={'model__hidden_layer_sizes':[(64,),(128,),(128,64),(256,128)],
 'model__alpha':loguniform(1e-5,1e-1), 'model__learning_rate_init':loguniform(1e-4,3e-3),
 'model__batch_size':[64,128,256]}
print(base_model)

Pipeline(steps=[('scale', StandardScaler()),
                ('model',
                 MLPClassifier(early_stopping=True, max_iter=240,
                               n_iter_no_change=15, random_state=42,
                               validation_fraction=0.15))])


In [5]:
def store_result(model_name,variant,pred,train_pred,best_params,cv_score=None,seconds=None):
    prefix=RESULT_DIR/f'{model_name}_{variant}'
    r={'Model':model_name,'Variant':variant,'CV_Macro_F1':cv_score,
       'Train_Macro_F1':float(f1_score(y[train_idx],train_pred,average='macro',zero_division=0)),
       'Val_Accuracy':float(accuracy_score(y[val_idx],pred)),
       'Val_Macro_F1':float(f1_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Precision':float(precision_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Recall':float(recall_score(y[val_idx],pred,average='macro',zero_division=0)),
       'MEL_Recall':float(recall_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'MEL_Precision':float(precision_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'Predicted_Classes':int(len(np.unique(pred))),'Best_Params':json.dumps(best_params,ensure_ascii=False,default=str),
       'Search_Seconds':seconds,'Train_N':len(train_idx),'Val_N':len(val_idx),'Test_N':len(test_idx)}
    pd.DataFrame({'image_id':ids[val_idx], 'actual':y[val_idx], 'predicted':pred}).to_csv(str(prefix)+'_val_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y[val_idx],pred,labels=classes),index=classes,columns=classes).to_csv(str(prefix)+'_confusion_matrix.csv')
    print(classification_report(y[val_idx],pred,labels=classes,zero_division=0))
    return r


In [7]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Sınıf eşleştirmeleri:")
for i, name in enumerate(encoder.classes_):
    print(i, "=", name)

Sınıf eşleştirmeleri:
0 = AKIEC
1 = BCC
2 = BKL
3 = DF
4 = MEL
5 = NV
6 = VASC


In [8]:
rows=[]
for variant,path in paths.items():
    started=time.time()
    X=np.load(path,mmap_mode='r',allow_pickle=False)
    Xt=np.asarray(X[train_idx],dtype=np.float32)
    Xv=np.asarray(X[val_idx],dtype=np.float32)
    assert np.isfinite(Xt).all() and np.isfinite(Xv).all()
    search=RandomizedSearchCV(base_model,param_space,n_iter=N_ITER,scoring='f1_macro',
      cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=42),n_jobs=SEARCH_JOBS,
      random_state=42,refit=True,verbose=1,error_score='raise')
    #search.fit(Xt,y[train_idx]);pred=search.predict(Xv);tr=search.predict(Xt)
    search.fit(Xt, y_encoded[train_idx])

    pred_encoded = search.predict(Xv)
    tr_encoded = search.predict(Xt)

    pred = encoder.inverse_transform(pred_encoded.astype(int))
    tr = encoder.inverse_transform(tr_encoded.astype(int))




    prefix=RESULT_DIR/f'{MODEL_NAME}_{variant}'
    pd.DataFrame(search.cv_results_).to_csv(str(prefix)+'_cv_results.csv',index=False)
    dump(search.best_estimator_,str(prefix)+'_model.joblib')
    r=store_result(MODEL_NAME,variant,pred,tr,search.best_params_,search.best_score_,round(time.time()-started,1))
    rows.append(r); print(variant, r['Val_Macro_F1'],search.best_params_)
    del Xt,Xv,X,search
summary=pd.DataFrame(rows).sort_values('Val_Macro_F1',ascending=False)
summary.to_csv(RESULT_DIR/f'{MODEL_NAME}_summary.csv',index=False)
display(summary)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
              precision    recall  f1-score   support

       AKIEC       0.42      0.22      0.29        49
         BCC       0.51      0.47      0.49        77
         BKL       0.50      0.45      0.47       165
          DF       0.33      0.11      0.17        18
         MEL       0.41      0.38      0.39       167
          NV       0.83      0.90      0.86      1006
        VASC       0.43      0.14      0.21        21

    accuracy                           0.73      1503
   macro avg       0.49      0.38      0.41      1503
weighted avg       0.71      0.73      0.71      1503

original 0.4134003137235694 {'model__alpha': np.float64(0.004013783023846251), 'model__batch_size': 64, 'model__hidden_layer_sizes': (128,), 'model__learning_rate_init': np.float64(0.0011654126622287456)}
Fitting 3 folds for each of 12 candidates, totalling 36 fits
              precision    recall  f1-score   support

       AKIEC       0.

,Model,Variant,CV_Macro_F1,Train_Macro_F1,Val_Accuracy,Val_Macro_F1,Val_Macro_Precision,Val_Macro_Recall,MEL_Recall,MEL_Precision,Predicted_Classes,Best_Params,Search_Seconds,Train_N,Val_N,Test_N
1,MLP_Color,masked,0.408325,0.746167,0.731204,0.432516,0.481995,0.407972,0.347305,0.467742,7,"{""model__alpha"": 1.7073967431528103e-05, ""mode...",202.2,7009,1503,1503
0,MLP_Color,original,0.365739,0.752851,0.729208,0.413400,0.491180,0.381902,0.377246,0.411765,7,"{""model__alpha"": 0.004013783023846251, ""model_...",179.8,7009,1503,1503
